# Training Data Extraction — Weights & Biases

Pull the logged **training** history for a single run from the `Food RL` wandb
project and save it to `results/training/` as a CSV.

Runs are named `Run_Seed{SEED}` (e.g. `Run_Seed42`). Set the seed in the config
cell below and run all cells.


## 1. Config

Set the seed you want to export. Everything else is derived from it.

In [1]:
# ── Config ──────────────────────────────────────────────────────────────
SEED = 2026                # <-- change this to the run you want to export

WANDB_ENTITY  = None           # None -> your default entity. Set to a string to override.
WANDB_PROJECT = "Food RL"      # project the runs were logged under
RUN_NAME      = f"Run_Seed{SEED}"

# Which logged keys to keep. Set to None to export ALL numeric history columns.
# These are the training metrics logged by agents/ppo.py during training.
METRIC_KEYS = [
    "train/reward",
    "train/reward_rolling_avg",
    "train/distance",
    "train/consumption",
    "train/actor_loss",
    "train/critic_loss",
    "train/entropy",
]

# Output location
from pathlib import Path
OUT_DIR = Path("results") / "training"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / f"{RUN_NAME}_training.csv"

print(f"Run     : {RUN_NAME}")
print(f"Project : {WANDB_PROJECT}")
print(f"Output  : {OUT_PATH}")


Run     : Run_Seed2026
Project : Food RL
Output  : results/training/Run_Seed2026_training.csv


## 2. Imports

Install `wandb` if it is not already available.

In [2]:
try:
    import wandb
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb"])
    import wandb

import pandas as pd

# Log in if needed. If you are already logged in (WANDB_API_KEY env var or a
# prior `wandb login`), this is a no-op. Otherwise it will prompt for a key.
wandb.login()
api = wandb.Api()


wandb: Currently logged in as: charithapalika (charitha_palika) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. Locate the run

Find the run whose display name matches `RUN_NAME`. If several runs share the
name (e.g. re-runs), the most recently created one is used.


In [3]:
def find_run(api, entity, project, run_name):
    """Return the wandb Run matching `run_name` (most recent if duplicated)."""
    path = f"{entity}/{project}" if entity else project
    runs = api.runs(path, filters={"display_name": run_name})
    runs = list(runs)
    if not runs:
        raise ValueError(
            f"No run named '{run_name}' found in '{path}'. "
            f"Check the seed, project name, and entity."
        )
    if len(runs) > 1:
        print(f"[warn] {len(runs)} runs named '{run_name}'; using most recent.")
        runs.sort(key=lambda r: r.created_at, reverse=True)
    run = runs[0]
    print(f"Found run: {run.name}  (id={run.id}, state={run.state})")
    return run


run = find_run(api, WANDB_ENTITY, WANDB_PROJECT, RUN_NAME)


Found run: Run_Seed2026  (id=721auofp, state=finished)


## 4. Download the full history

`run.scan_history()` streams every logged step without the default 500-row
sampling that `run.history()` applies, so the CSV contains the complete
training curve.


In [4]:
def fetch_history(run, metric_keys=None):
    """Download full logged history as a DataFrame.

    metric_keys : list of keys to keep (plus step/timestamp columns),
                  or None to keep every logged column.
    """
    if metric_keys is None:
        keys = None  # scan_history returns all keys
    else:
        # Always include step/time bookkeeping columns if present.
        keys = list(dict.fromkeys(metric_keys + ["_step", "_runtime", "_timestamp"]))

    rows = list(run.scan_history(keys=keys))
    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("Run has no logged history.")

    # Order columns: step first, then requested metrics, then the rest.
    front = [c for c in ["_step", "_runtime", "_timestamp"] if c in df.columns]
    if metric_keys is not None:
        metrics = [c for c in metric_keys if c in df.columns]
        rest    = [c for c in df.columns if c not in front + metrics]
        df = df[front + metrics + rest]
        missing = [k for k in metric_keys if k not in df.columns]
        if missing:
            print(f"[warn] keys not found in run history: {missing}")

    df = df.sort_values("_step").reset_index(drop=True) if "_step" in df.columns else df
    return df


df = fetch_history(run, METRIC_KEYS)
print(f"Rows: {len(df)}   Columns: {list(df.columns)}")
df.head()


Rows: 1002   Columns: ['_step', '_runtime', '_timestamp', 'train/reward', 'train/reward_rolling_avg', 'train/distance', 'train/consumption', 'train/actor_loss', 'train/critic_loss', 'train/entropy']


,_step,_runtime,_timestamp,train/reward,train/reward_rolling_avg,train/distance,train/consumption,train/actor_loss,train/critic_loss,train/entropy
0,2,0.247814,1.785045e+09,-844.387182,-1278.001718,1373.494656,105,-0.012744,1667.009369,0.655738
1,4,0.247814,1.785045e+09,-507.944220,-896.287187,1073.410867,73,-0.013005,413.857859,0.545085
2,6,0.247814,1.785045e+09,-15.245348,-611.407569,406.391470,37,-0.008465,8.784166,0.420367
3,8,0.247814,1.785045e+09,-6.206738,-465.983859,369.264886,33,-0.011600,6.149622,0.291749
4,10,0.247814,1.785045e+09,-62.745958,-366.621458,501.672756,39,-0.004453,12.448032,0.427174


## 5. Save to CSV

In [5]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df)} rows -> {OUT_PATH}")


Saved 1002 rows -> results/training/Run_Seed2026_training.csv


---
### Export several seeds at once (optional)

Uncomment and run to loop over multiple seeds in one pass.


In [6]:
# for seed in [0, 42, 64, 100]:
#     name = f"Run_Seed{seed}"
#     r = find_run(api, WANDB_ENTITY, WANDB_PROJECT, name)
#     d = fetch_history(r, METRIC_KEYS)
#     out = OUT_DIR / f"{name}_training.csv"
#     d.to_csv(out, index=False)
#     print(f"{name}: {len(d)} rows -> {out}")
